In [ ]:
import Pkg
Pkg.activate(".")  
Pkg.add("FFTW")

In [ ]:
using SerialPorts
using FFTW
using Statistics
using DSP
using Printf
using Plots

In [ ]:
list_serialports()

In [ ]:
# Constants
const NUM_SAMPLES = 1024
const REF_RESISTANCE = 600.250  # Ohms
const ADC_VREF = 3.3              # Volts
const ADC_MAX_VALUE = 4095.0      # 12-bit ADC
const NUM_CALIBRATION_SAMPLES = 30

In [ ]:
const START_MARKER = 0xAA
const END_MARKER = 0x55

In [ ]:
@enum CircuitModel begin
    SERIES_RC = 1
    SERIES_RL = 2
    PARALLEL_RC = 3
    PARALLEL_RL = 4
    CUSTOM = 5
end

In [ ]:
mutable struct CircuitConfig
    model::CircuitModel
    frequency::Float64
    # For custom models
    has_series_r::Bool
    has_series_l::Bool
    has_series_c::Bool
    has_parallel_r::Bool
    has_parallel_l::Bool
    has_parallel_c::Bool
end

In [ ]:
CircuitConfig() = CircuitConfig(SERIES_RC, 1000.0, true, false, true, false, false, false)

In [ ]:
mutable struct OnlineStats
    mean::Float64
    M2::Float64
    count::Int64
end

OnlineStats() = OnlineStats(0.0, 0.0, 0)


In [ ]:
# Global statistics variables
impedance_stats = OnlineStats()
phase_stats = OnlineStats()
real_z_stats = OnlineStats()
imag_z_stats = OnlineStats()
v1_stats = OnlineStats()
v2_stats = OnlineStats()
calibration_count = 0
current_circuit = CircuitConfig()

In [ ]:
function update_stats!(stats::OnlineStats, value::Float64)
    stats.count += 1
    delta = value - stats.mean
    stats.mean += delta / stats.count
    delta2 = value - stats.mean
    stats.M2 += delta * delta2
end


In [ ]:
function std_dev(stats::OnlineStats)
    stats.count < 2 && return 0.0
    return sqrt(stats.M2 / (stats.count - 1))
end

function std_err(stats::OnlineStats)
    stats.count < 2 && return 0.0
    return sqrt(stats.M2 / (stats.count - 1)) / sqrt(Float64(stats.count))
end

In [ ]:
function prepare_fft_data(adc_data::Vector{UInt16})
    # Convert ADC counts to voltages
    VREF = 3.3
    ADC_MAX = 4095.0
    voltages = VREF * (Float64.(adc_data) ./ ADC_MAX)
    
    # Remove DC offset
    voltages_ac = voltages .- mean(voltages)
    
    # Apply window (e.g., Hann window)
    window = hann(length(voltages_ac))
    windowed_data = voltages_ac .* window
    
    return windowed_data, window
end

In [ ]:

function extract_signal_parameters(fft_complex::Vector{ComplexF64}, bin_index::Int, window::Vector{Float64})
    # Clamp bin index
    bin_index = max(1, min(bin_index + 1, length(fft_complex)))
    component = fft_complex[bin_index]
    real_part, imag_part = real(component), imag(component)

    # Window correction factor
    window_gain = sum(window) / NUM_SAMPLES

    # Calculate amplitude
    magnitude_lsb = (2.0 * abs(component)) / (NUM_SAMPLES * window_gain)
    magnitude_v = magnitude_lsb * (ADC_VREF / ADC_MAX_VALUE)

    # Calculate phase
    phase_rad = atan(imag_part, real_part)

    return magnitude_v, phase_rad
end

In [ ]:

function compute_fft(data::Vector{Float64})
    # Perform FFT
    fft_result = fft(data)
    return fft_result
end

In [ ]:
function calculate_impedance(mag_v1::Float64, phase_v1::Float64, 
                            mag_v2::Float64, phase_v2::Float64)
    # Voltage ratio
    voltage_ratio = mag_v2 / mag_v1
    
    # Phase difference (V2 - V1)
    phase_diff = phase_v2 - phase_v1
    
    # Normalize phase to [-π, π]
    while phase_diff > π
        phase_diff -= 2π
    end
    while phase_diff < -π
        phase_diff += 2π
    end
    
    # Calculate I and Q components
    I_component = mag_v2 * cos(phase_diff)
    Q_component = mag_v2 * sin(phase_diff)
    
    # Z(ω) = -Ri × (V2/V1)
    impedance_mag = REF_RESISTANCE * voltage_ratio
    
    # Phase (add 180° for the negative sign)
    impedance_phase = phase_diff + π
    
    # Normalize phase
    while impedance_phase > π
        impedance_phase -= 2π
    end
    while impedance_phase < -π
        impedance_phase += 2π
    end
    
    # Convert to rectangular form
    real_z = impedance_mag * cos(impedance_phase)
    imag_z = impedance_mag * sin(impedance_phase)
    
    return impedance_mag, impedance_phase, real_z, imag_z, I_component, Q_component
end

In [ ]:
function extract_circuit_parameters(real_z::Float64, imag_z::Float64, frequency::Float64, config::CircuitConfig)
    ω = 2π * frequency
    
    if config.model == SERIES_RC
        Rs = real_z
        Cs = imag_z < 0 ? -1.0 / (ω * imag_z) : 0.0
        return Dict("Rs" => Rs, "Cs" => Cs * 1e6)  # Convert to µF
        
    elseif config.model == SERIES_RL
        Rs = real_z
        Ls = imag_z > 0 ? imag_z / ω : 0.0
        return Dict("Rs" => Rs, "Ls" => Ls * 1e3)  # Convert to mH
        
    elseif config.model == PARALLEL_RC
        # Z = (R * (1/(jωC))) / (R + 1/(jωC))
        # Real and imaginary parts allow solving for R and C
        if abs(real_z) > 1e-6
            G = real_z / (real_z^2 + imag_z^2)  # Conductance
            B = -imag_z / (real_z^2 + imag_z^2)  # Susceptance
            Rp = 1.0 / G
            Cp = B / ω
            return Dict("Rp" => Rp, "Cp" => Cp * 1e6)  # Convert to µF
        end
        
    elseif config.model == PARALLEL_RL
        if abs(real_z) > 1e-6
            G = real_z / (real_z^2 + imag_z^2)
            B = -imag_z / (real_z^2 + imag_z^2)
            Rp = 1.0 / G
            Lp = -1.0 / (ω * B)
            return Dict("Rp" => Rp, "Lp" => Lp * 1e3)  # Convert to mH
        end
        
    elseif config.model == CUSTOM
        # For custom models, return raw impedance components
        result = Dict{String, Float64}()
        if config.has_series_r || config.has_series_l || config.has_series_c
            result["Z_real"] = real_z
            result["Z_imag"] = imag_z
        end
        return result
    end
    
    return Dict{String, Float64}()
end

In [ ]:
function display_circuit_parameters(params::Dict{String, Float64})
    println("\n  Circuit Parameters:")
    for (key, value) in params
        if endswith(key, "s") && key != "Rs"
            # Capacitance or Inductance
            if startswith(key, "C")
                println("    $key = $(round(value, digits=3)) µF")
            elseif startswith(key, "L")
                println("    $key = $(round(value, digits=3)) mH")
            end
        elseif endswith(key, "p") && key != "Rp"
            if startswith(key, "C")
                println("    $key = $(round(value, digits=3)) µF")
            elseif startswith(key, "L")
                println("    $key = $(round(value, digits=3)) mH")
            end
        else
            println("    $key = $(round(value, digits=2)) Ω")
        end
    end
end

In [ ]:
function plot_measurement_analysis(adc1_data::Vector{UInt16}, adc3_data::Vector{UInt16}, 
                                   fft_v1, fft_v2, signal_bin, fsampling,
                                   z_mag, z_phase, real_z, imag_z, freq_out,
                                   measurement_num::Int)
    
    
    # Create time axis
    t = (0:NUM_SAMPLES-1) ./ Float64(fsampling)
    t_ms = t .* 1000  # Convert to milliseconds
    
    # Convert ADC to voltages for plotting
    VREF = 3.3
    ADC_MAX = 4095.0
    adc1_volts = VREF * (Float64.(adc1_data) ./ ADC_MAX)
    adc3_volts = VREF * (Float64.(adc3_data) ./ ADC_MAX)
    
    # Create frequency axis for FFT
    freq_axis = (0:NUM_SAMPLES-1) .* (fsampling / NUM_SAMPLES)
    freq_axis_khz = freq_axis ./ 1000  # Convert to kHz
    
    # Calculate impedance components
    capacitance = -1.0 / (2π * freq_out * imag_z)  # In Farads
    capacitance_nf = capacitance * 1e9  # Convert to nanofarads
    
    # Create a 2x3 subplot layout
    p = plot(layout=(2,3), size=(1800, 1000), plot_title="Measurement #$measurement_num @ $(freq_out) Hz")
    
    # Plot 1: Time domain waveforms
    plot!(p[1], t_ms, adc1_volts, 
          label="V1 (ADC1)", 
          xlabel="Time (ms)", 
          ylabel="Voltage (V)", 
          title="Time Domain Signals",
          linewidth=1.5,
          color=:blue)
    plot!(p[1], t_ms, adc3_volts, 
          label="V2 (ADC3)", 
          linewidth=1.5,
          color=:red)
    
    # Plot 2: FFT Magnitude (full spectrum)
    fft_range = 1:NUM_SAMPLES÷2
    plot!(p[2], freq_axis_khz[fft_range], abs.(fft_v1[fft_range]),
          label="V1 FFT",
          xlabel="Frequency (kHz)",
          ylabel="Magnitude",
          title="FFT Magnitude Spectrum",
          linewidth=1.5,
          yscale=:log10,
          color=:blue)
    plot!(p[2], freq_axis_khz[fft_range], abs.(fft_v2[fft_range]),
          label="V2 FFT",
          linewidth=1.5,
          color=:red)
    # Mark the signal frequency
    vline!(p[2], [freq_out/1000], label="Signal ($(freq_out) Hz)", 
           linestyle=:dash, color=:green, linewidth=2)
    
    # Plot 3: FFT Zoom around signal frequency
    zoom_bins = 20  # Show ±20 bins around signal
    bin_start = max(1, signal_bin - zoom_bins)
    bin_end = min(NUM_SAMPLES÷2, signal_bin + zoom_bins)
    zoom_range = bin_start:bin_end
    
    plot!(p[3], freq_axis[zoom_range], abs.(fft_v1[zoom_range]),
          label="V1 FFT",
          xlabel="Frequency (Hz)",
          ylabel="Magnitude",
          title="FFT Zoom @ Signal Frequency",
          linewidth=2,
          marker=:circle,
          markersize=3,
          color=:blue)
    plot!(p[3], freq_axis[zoom_range], abs.(fft_v2[zoom_range]),
          label="V2 FFT",
          linewidth=2,
          marker=:circle,
          markersize=3,
          color=:red)
    vline!(p[3], [freq_out], label="Signal Bin", 
           linestyle=:dash, color=:green, linewidth=2)
    
    # Plot 4: Impedance Phasor (Complex Plane)
    plot!(p[4], [0, real_z], [0, imag_z],
          arrow=true,
          linewidth=3,
          color=:purple,
          label="Z = $(round(z_mag, digits=1))Ω ∠$(round(rad2deg(z_phase), digits=1))°",
          xlabel="Real (Ω)",
          ylabel="Imaginary (Ω)",
          title="Impedance Phasor",
          aspect_ratio=:equal,
          legend=:topright)
    scatter!(p[4], [real_z], [imag_z], 
            markersize=8, 
            color=:purple,
            label="")
    # Add axes
    hline!(p[4], [0], color=:black, linestyle=:dash, linewidth=0.5, label="")
    vline!(p[4], [0], color=:black, linestyle=:dash, linewidth=0.5, label="")
    # Add annotations
    annotate!(p[4], real_z*0.5, imag_z*0.5, 
             text("R=$(round(real_z, digits=1))Ω\nX=$(round(imag_z, digits=1))Ω", 10))
    
    # Plot 5: RC Circuit Model
    # equivalent series RC circuit
    plot!(p[5], 
          xlims=(0, 10), ylims=(0, 10),
          title="Equivalent Series RC Circuit",
          xlabel="",
          ylabel="",
          showaxis=false,
          grid=false,
          legend=false)
    
    # circuit schematic 
    # Resistor
    plot!(p[5], [1, 3], [5, 5], linewidth=3, color=:black)
    annotate!(p[5], 2, 6, text("Rs\n$(round(real_z, digits=1)) Ω", 10, :center))
    
    # Capacitor
    plot!(p[5], [5, 5], [4, 6], linewidth=3, color=:black)
    plot!(p[5], [5.2, 5.2], [4, 6], linewidth=3, color=:black)
    annotate!(p[5], 5.1, 7, text("Cs\n$(round(abs(capacitance_nf), digits=2)) nF", 10, :center))
    
    # Connections
    plot!(p[5], [3, 5], [5, 5], linewidth=2, color=:black)
    plot!(p[5], [5.2, 7], [5, 5], linewidth=2, color=:black)
    plot!(p[5], [0.5, 1], [5, 5], linewidth=2, color=:black)
    plot!(p[5], [7, 9], [5, 5], linewidth=2, color=:black)
    
    # Terminals
    scatter!(p[5], [0.5, 9], [5, 5], markersize=8, color=:red, label="")
    annotate!(p[5], 0.5, 4, text("V1", 10))
    annotate!(p[5], 9, 4, text("V2", 10))
    
    # impedance info
    annotate!(p[5], 5, 2, 
             text("@ $(freq_out) Hz\n|Z| = $(round(z_mag, digits=1)) Ω\n∠Z = $(round(rad2deg(z_phase), digits=1))°", 9, :center))
    
    # Plot 6: Impedance Components Bar Chart
    bar!(p[6], ["Resistance\n(R)", "Reactance\n(X)", "Magnitude\n(|Z|)"],
         [abs(real_z), abs(imag_z), z_mag],
         xlabel="Component",
         ylabel="Impedance (Ω)",
         title="Impedance Components",
         legend=false,
         color=[:green, :orange, :purple],
         bar_width=0.6)
    
    # value labels on bars
    annotate!(p[6], 1, abs(real_z)*1.05, text("$(round(real_z, digits=1)) Ω", 9))
    annotate!(p[6], 2, abs(imag_z)*1.05, text("$(round(imag_z, digits=1)) Ω", 9))
    annotate!(p[6], 3, z_mag*1.05, text("$(round(z_mag, digits=1)) Ω", 9))
    
    display(p)
    
    return p
end

In [ ]:
function process_measurement(adc1_data::Vector{UInt16}, adc3_data::Vector{UInt16}, 
                            freq_out::UInt32, fsampling::UInt32; 
                            show_plots::Bool=false,
                            measurement_num::Int=0)
    signal_bin = Int(clamp(round((freq_out * NUM_SAMPLES) / fsampling), 1, NUM_SAMPLES÷2))

    println("Processing measurement:")
    println("  Signal frequency: $(freq_out) Hz")
    println("  FFT bin: $(signal_bin)")

    fft_input_v1, window = prepare_fft_data(adc1_data)
    fft_input_v2, _ = prepare_fft_data(adc3_data)

    fft_v1 = compute_fft(fft_input_v1)
    fft_v2 = compute_fft(fft_input_v2)

    mag_v1, phase_v1 = extract_signal_parameters(fft_v1, signal_bin, window)
    mag_v2, phase_v2 = extract_signal_parameters(fft_v2, signal_bin, window)

    z_mag, z_phase, real_z, imag_z, I_comp, Q_comp =
        calculate_impedance(mag_v1, phase_v1, mag_v2, phase_v2)

    global calibration_count
    if calibration_count < NUM_CALIBRATION_SAMPLES
        update_stats!(impedance_stats, z_mag)
        update_stats!(phase_stats, z_phase)
        update_stats!(real_z_stats, real_z)
        update_stats!(imag_z_stats, imag_z)
        update_stats!(v1_stats, mag_v1)
        update_stats!(v2_stats, mag_v2)
        calibration_count += 1
        println("  [$(calibration_count)/$(NUM_CALIBRATION_SAMPLES)] |Z|=$(round(z_mag,digits=3)) Ω (Mean=$(round(impedance_stats.mean,digits=3)), Std=$(round(std_dev(impedance_stats),digits=3)))")
    end

    # Circuit parameter extraction
    circuit_params = extract_circuit_parameters(real_z, imag_z, Float64(freq_out), current_circuit)

    println("  V1: $(round(mag_v1,digits=6)) V @ $(round(rad2deg(phase_v1),digits=2))°")
    println("  V2: $(round(mag_v2,digits=6)) V @ $(round(rad2deg(phase_v2),digits=2))°")
    println("  |Z|: $(round(z_mag,digits=3)) Ω, ∠Z: $(round(rad2deg(z_phase),digits=2))°")
    println("  Z = $(round(real_z,digits=3)) + $(round(imag_z,digits=3))j Ω")

    display_circuit_parameters(circuit_params)

    if show_plots
        plot_measurement_analysis(adc1_data, adc3_data, fft_v1, fft_v2, 
                                 signal_bin, fsampling, z_mag, z_phase, 
                                 real_z, imag_z, freq_out, measurement_num)
    end

    return z_mag, z_phase, real_z, imag_z, mag_v1, mag_v2, circuit_params
end

In [ ]:
function print_final_statistics()
    println("\n" * "="^80)
    println("                    CALIBRATION COMPLETE")
    println("Samples Collected: $(impedance_stats.count)")
    println("-"^80)
    
    println("\nImpedance Magnitude:")
    println("  Mean:    $(impedance_stats.mean) Ohms")
    println("  Std Dev: $(std_dev(impedance_stats)) Ohms")
    println("  StdErr:  $(std_err(impedance_stats)) Ohms")
    println("  CV:      $((std_dev(impedance_stats) / impedance_stats.mean) * 100)%")
    
    println("\nPhase:")
    println("  Mean:    $(rad2deg(phase_stats.mean)) degrees")
    println("  Std Dev: $(rad2deg(std_dev(phase_stats))) degrees")
    
    println("\nAveraged Impedance:")
    println("  Z̄ = $(real_z_stats.mean) + $(imag_z_stats.mean)j Ohms")
    println("="^80 * "\n")
end

In [ ]:


function select_circuit_model()
    println("\n" * "="^60)
    println("SELECT CIRCUIT MODEL")
    println("="^60)
    println("1. Series RC (Rs + Cs)")
    println("2. Series RL (Rs + Ls)")
    println("3. Parallel RC (Rp || Cp)")
    println("4. Parallel RL (Rp || Lp)")
    println("5. Custom (specify components)")
    println("="^60)
    print("Enter choice (1-5): ")
    
    choice = parse(Int, readline())
    
    if choice == 1
        current_circuit.model = SERIES_RC
        println("Selected: Series RC model")
    elseif choice == 2
        current_circuit.model = SERIES_RL
        println("Selected: Series RL model")
    elseif choice == 3
        current_circuit.model = PARALLEL_RC
        println("Selected: Parallel RC model")
    elseif choice == 4
        current_circuit.model = PARALLEL_RL
        println("Selected: Parallel RL model")
    elseif choice == 5
        configure_custom_model()
    else
        println("Invalid choice, defaulting to Series RC")
        current_circuit.model = SERIES_RC
    end
    
    print("Enter test frequency (Hz): ")
    freq = parse(Float64, readline())
    current_circuit.frequency = freq
    
    println("\nConfiguration complete!")
    println("Model: $(current_circuit.model)")
    println("Frequency: $(current_circuit.frequency) Hz\n")
end



In [ ]:
function configure_custom_model()
    current_circuit.model = CUSTOM
    println("\nConfigure custom circuit model:")
    print("Include series resistance? (y/n): ")
    current_circuit.has_series_r = lowercase(strip(readline())) == "y"
    print("Include series inductance? (y/n): ")
    current_circuit.has_series_l = lowercase(strip(readline())) == "y"
    
    print("Include series capacitance? (y/n): ")
    current_circuit.has_series_c = lowercase(strip(readline())) == "y"
    
    print("Include parallel resistance? (y/n): ")
    current_circuit.has_parallel_r = lowercase(strip(readline())) == "y"
    
    print("Include parallel inductance? (y/n): ")
    current_circuit.has_parallel_l = lowercase(strip(readline())) == "y"
    print("Include parallel capacitance? (y/n): ")
    current_circuit.has_parallel_c = lowercase(strip(readline())) == "y"
end

println("Impedance Meter")
println("==========================================\n")

select_circuit_model()

In [ ]:
function receive_data_from_stm32(port)
    # Waiting for the data to be available
    timeout = 10.0  # 10 second timeout
    start_time = time()
    
    expected_bytes = 4106  # 1 start + 4 freq + 4 fs + 2048 ADC1 + 2048 ADC3 + 1 end
    
    while bytesavailable(port) < expected_bytes
        if (time() - start_time) > timeout
            error("Timeout waiting for data from STM32 (got $(bytesavailable(port)) bytes, expected $expected_bytes)")
        end
        sleep(0.01)
    end
    
    # Read all available data
    raw = readavailable(port)
    println("Received $(length(raw)) bytes")
    
    # Convert to Vector{UInt8} 
    raw = Vector{UInt8}(raw)
    
    # Verify markers
    if raw[1] != 0xAA
        @warn @sprintf(" Start marker missing or incorrect: 0x%02X", raw[1])
    end
    if raw[end] != 0x55
        @warn @sprintf(" End marker missing or incorrect: 0x%02X", raw[end])
    end
    
    # Parse metadata
    freq_out = reinterpret(UInt32, raw[2:5])[1]
    fsampling = reinterpret(UInt32, raw[6:9])[1]
    @printf("freq_out = %d Hz, fsampling = %d Hz\n", freq_out, fsampling)
    
    # Extract ADC buffers
    num_samples = 1024
    adc1_bytes = raw[10 : 10 + num_samples*2 - 1]
    adc3_bytes = raw[10 + num_samples*2 : 10 + num_samples*4 - 1]
    
    adc1_data = Vector{UInt16}(reinterpret(UInt16, adc1_bytes))
    adc3_data = Vector{UInt16}(reinterpret(UInt16, adc3_bytes))
    
    
    # Convert to voltages for display only
    VREF = 3.3
    ADC_MAX = 4095.0
    adc1_volts = VREF * (Float64.(adc1_data) ./ ADC_MAX)
    adc3_volts = VREF * (Float64.(adc3_data) ./ ADC_MAX)
    
    println(" Parsed $(length(adc1_data)) samples per channel.")
    println("First 10 ADC1 samples (V): ", round.(adc1_volts[1:10], digits=4))
    println("First 10 ADC3 samples (V): ", round.(adc3_volts[1:10], digits=4))
    
    # Return raw UInt16 data for process_measurement
    return adc1_data, adc3_data, freq_out, fsampling
end

In [ ]:
function send_results_to_stm32(port, z_mag::Float64, z_phase::Float64, 
                               real_z::Float64, imag_z::Float64,
                               mag_v1::Float64, mag_v2::Float64, stderr::Float64)

    write(port, START_MARKER)

    # Send model ID right after start marker
    model_id = UInt8(current_circuit.model)
    write(port, model_id)

    # Send 7 results (Float32)
    result_data = Float32[z_mag, z_phase, real_z, imag_z, mag_v1, mag_v2, stderr]
    write(port, reinterpret(UInt8, result_data))

    write(port, END_MARKER)
    flush(port)
end


In [ ]:
function main(max_measurements=nothing)
    port_name = "COM3"
    println("Opening serial port: $(port_name)")
    
    port = SerialPort(port_name, 115200)
    measurement_count = 0
    
    try
        # Wait for port to stabilize
        sleep(10)
        
        # Flush any garbage in the buffer
        println("Flushing serial buffer...")
        while bytesavailable(port) > 0
            read(port, bytesavailable(port))
        end
        sleep(40)
        
        println("\nWaiting for measurements from STM32...")
        println("Press Ctrl+C to cancel\n")
        
        while true
            try
                # Receive data from STM32
                adc1_data, adc3_data, freq_out, fsampling = receive_data_from_stm32(port)
                
                # Process the measurement
                z_mag, z_phase, real_z, imag_z, mag_v1, mag_v2, _ = 
                    process_measurement(adc1_data, adc3_data, freq_out, fsampling, 
                       show_plots=true,  # Enable plots
                       measurement_num=measurement_count+1)
                
                # Get current standard error
                stderr_val = std_err(impedance_stats)
                
                # Send results back to STM32
                send_results_to_stm32(port, z_mag, z_phase, real_z, imag_z, 
                                     mag_v1, mag_v2, stderr_val)
                
                measurement_count += 1
                println("Results sent to STM32 [Measurement #$(measurement_count)]\n")
                
                # Stop after max_measurements if specified
                if max_measurements !== nothing && measurement_count >= max_measurements
                    println("\nCompleted $(max_measurements) measurements")
                    break
                end
                
            catch e
                if isa(e, EOFError) || isa(e, Base.IOError)
                    println("Connection lost, waiting for reconnection...")
                    sleep(1)
                elseif isa(e, InterruptException)
                    println("\nInterrupted by user")
                    break
                else
                    println(" Error: ", e)
                    # Print stack trace for debugging
                    for (exc, bt) in Base.catch_stack()
                        showerror(stdout, exc, bt)
                        println()
                    end
                    break
                end
            end
        end
    finally
        close(port)
        println("Serial port closed")
    end
end

In [ ]:
 main(20) 

In [ ]:
using Dates
timestamp = Dates.format(now(), "yyyymmdd_HHMMSS")
filename = "measurement_" * timestamp * ".txt"
open(filename, "a") do io
    redirect_stdout(io) do
        println("\n-- New measurement batch --")
        main(5)
    end
end